# Five-epoch training + full 240-task decoding speed test

This timing notebook starts fresh from the original verified epoch-330
pretrained checkpoint, trains exactly **five epochs on all 240 tasks /
726 demonstration pairs**, then decodes **all 336 queries with 80
augmentations each: 26,880 augmented cases**. Default **SMOKE_TEST=False**;
there is no four-task subset in this benchmark.

Attach [timing240](https://www.kaggle.com/datasets/mithilavakde/mdlarc-arc2-timing240-v1),
[epoch330 checkpoint](https://www.kaggle.com/datasets/mithilavakde/mdlarc-600m-epoch330),
and [FA2 wheel](https://www.kaggle.com/datasets/mithilavakde/flash-attn-cp313-torch211-cu128).
Select **L4 × 4**, **Internet off**, restart the session and **Run All**.

Only blocks **20–23** train; task embeddings stay disabled. Training is
microbatch **2/GPU**, accumulation **2/GPU**, effective batch **16**,
NorMuon **0.0003**, AdamW **0.00028**, training augmentations **80**.
Four independent inference workers use **batch 1/GPU**, prefill budget
**4096**, and **native four-head speculative decoding enabled**.

The final cell prints **TIMING BENCHMARK COMPLETE**, measured training
time, full decoding wall time and aggregate augmented-cases/second.
Decode time includes process startup, checkpoint loading, compilation,
generation, AAIVR and merging; post-generation answer scoring is separate.
These are measurements for five-epoch-adapted weights; prediction lengths
and decoding duration can change after 330 adaptation epochs.

Outputs: `timing240_benchmark.json`, `timing240_run_info.json`,
`timing240_predictions.json`, `timing240_scores.json`, plus per-GPU logs
and memory reports in `runs/last4_val240_timing_e5/`. Original40 and synthetic200
scores remain separate. Query solutions are opened only after decoding
and merging. **This notebook is a timing test, not a competition submission.**


In [ ]:
CHECKPOINT_IDENTITY = {'filename': 'time_arc2_600m_val240_last4_e5.ipynb', 'title': '600M last four blocks — five epochs and full timing240 decode', 'session': 'vast-ddp8-b200-b24a1-lr061-e70-noon-20261025-r5', 'epoch': 330, 'size': 5726752055, 'sha256': 'a17e48c09c7d4c99755faa4a5eaa02d96cda88f137b1471194d1ffd20b661582', 'augmentation_sampling_mode': 'context_palette', 'config': {'d_model': 1536, 'd_ff': 4096, 'n_layers': 24, 'n_heads': 24, 'num_task_embedding_latents': 3072, 'task_embeddings': False, 'attention_pattern': 'pair_group', 'split_target_demo_attention': True, 'n_predict': 4}, 'num_gpus': 4, 'preserve_runtime': True, 'train_first_layers': 0, 'train_last_layers': 4, 'epochs': 5, 'batch_size': 2, 'accumulation': 2, 'eval_batch_size': 1, 'normuon_lr': 0.0003, 'adamw_lr': 0.00028, 'training_demo_schedule': 'all', 'inference_demo_policy': 'leave_one_out', 'recipe': 'last4', 'runtime_source': 'kaggle/submit_arc2_600m.ipynb', 'runtime_source_sha256': '71ad055eb93ce682d470d695d95dd98bacbd375a2799774b08c419aea10a23ba', 'prefill_token_budget': 4096, 'inference_speculative': True, 'smoke_test': False, 'validation': {'dataset_ref': 'mithilavakde/mdlarc-arc2-timing240-v1', 'tasks': 240, 'demonstration_pairs': 726, 'query_pairs': 336, 'challenges': {'filename': 'arc2_val240_challenges.json', 'size': 1951766, 'sha256': '4692772246e8778ca30dc2cadb9d9807b0ad580fa8f837340095fca2f07f2813'}, 'solutions': {'filename': 'arc2_val240_solutions.json', 'size': 352752, 'sha256': '551d9d2fe02194a7d924123c394a94ec161410f24159e5ce7f3909b140bedfd2'}, 'manifest': {'filename': 'timing_manifest.json', 'size': 1250346, 'sha256': 'b051f8b0eda100f09356ada6f5339e6532cbdd64bbc72d7bb1cac5b51b4af826'}}, 'timing_benchmark': True, 'run_name': 'last4_val240_timing_e5', 'output_prefix': 'timing240'}
import os
import sys
from pathlib import Path

NUM_GPUS = 4
if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    if sys.modules["torch"].cuda.device_count() != NUM_GPUS:
        raise RuntimeError("Restart the session before selecting all four GPUs.")
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(map(str, range(NUM_GPUS)))
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if int(os.environ.get("WORLD_SIZE", "1")) != 1 or "LOCAL_RANK" in os.environ:
    raise RuntimeError("Run the notebook directly; it launches its own torchrun workers.")

CHECKPOINT_PATH = None
TEST_CHALLENGES = None
ATTENTION_BACKEND = "flash"
FLASH_ATTN_WHEEL = None
ENABLE_COMPILE = True
TRAIN_LOG_MODE = "step"  # Log every optimizer update, including epoch/step/loss.
PROGRESS_INTERVAL_SECONDS = 30  # Report quiet worker processes without claiming training progress.
SMOKE_TEST = False  # Timing benchmark: keep False for five epochs and full240 / 80-augs decode.
EPOCHS = 5
TRAIN_FIRST_LAYERS = 0
TRAIN_LAST_LAYERS = 4
TRAIN_BATCH_SIZE = 2  # Per GPU.
GRADIENT_ACCUMULATION_STEPS = 2
EVAL_BATCH_SIZE = 1  # Per GPU; raise after checking memory.
TRAIN_AUGMENTS = 80
EVAL_AUGMENTS = 80
NUM_WORKERS = 0  # Four training processes already supply CPU parallelism.
NORMUON_LR = 0.0003
ADAMW_LR = 0.00028
MAX_FINETUNE_HOURS = 8.0

if ATTENTION_BACKEND not in {"flash", "native"}:
    raise ValueError("Choose flash or native attention.")
if not ENABLE_COMPILE:
    os.environ["TORCH_COMPILE_DISABLE"] = "1"
else:
    os.environ.pop("TORCH_COMPILE_DISABLE", None)
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
os.chdir(WORKING)

os.environ.pop("MDLARC_CHECKPOINT_QUEUE", None)
os.environ.pop("MDLARC_RUN_DIR", None)
print("Mode:", "SMOKE TEST" if SMOKE_TEST else "FULL VALIDATION")
print("Training GPUs:", NUM_GPUS, "full effective batch:",
      NUM_GPUS * TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

VALIDATION_IDENTITY = {'dataset_ref': 'mithilavakde/mdlarc-arc2-timing240-v1', 'tasks': 240, 'demonstration_pairs': 726, 'query_pairs': 336, 'challenges': {'filename': 'arc2_val240_challenges.json', 'size': 1951766, 'sha256': '4692772246e8778ca30dc2cadb9d9807b0ad580fa8f837340095fca2f07f2813'}, 'solutions': {'filename': 'arc2_val240_solutions.json', 'size': 352752, 'sha256': '551d9d2fe02194a7d924123c394a94ec161410f24159e5ce7f3909b140bedfd2'}, 'manifest': {'filename': 'timing_manifest.json', 'size': 1250346, 'sha256': 'b051f8b0eda100f09356ada6f5339e6532cbdd64bbc72d7bb1cac5b51b4af826'}}

PREFILL_TOKEN_BUDGET = 4096
INFERENCE_SPECULATIVE = True
PINNED_SOURCE_NOTEBOOK_SHA256 = '71ad055eb93ce682d470d695d95dd98bacbd375a2799774b08c419aea10a23ba'

if SMOKE_TEST:
    raise ValueError("Keep SMOKE_TEST=False for the requested five-epoch full240 timing benchmark.")


## Offline CUDA / FlashAttention environment

Verify four L4 GPUs and the printed Python/PyTorch/CUDA/ABI against the
attached wheel. Keep FlashAttention 2 and torch.compile enabled for this
comparison. A changed Kaggle image may require a new compatible wheel.
Native attention is an explicit alternative and changes the runtime comparison.


In [ ]:
import importlib.util
import platform
import subprocess
import torch

# Available before extracting mdlARC, including in an Internet-off notebook.
import hashlib
import json
import shutil

def restore_wheel_filename(source):
    source = Path(source)
    manifest_path = source.parent / "wheel-manifest.json"
    if not manifest_path.is_file():
        return source
    manifest = json.loads(manifest_path.read_text())
    name = manifest["wheel"]
    if Path(name).name != name or not name.endswith(".whl"):
        raise ValueError("Invalid wheel filename in build manifest.")
    with source.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    if digest != manifest["wheel_sha256"]:
        raise ValueError("FlashAttention wheel checksum failed.")
    target = WORKING / name
    if source.resolve() != target.resolve():
        shutil.copyfile(source, target)
    return target

print("Python:", sys.version)
print("Platform:", platform.platform(), platform.machine())
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
print("C++ ABI:", getattr(torch._C, "_GLIBCXX_USE_CXX11_ABI", "unknown"))
if not torch.cuda.is_available():
    raise RuntimeError("Select GPU L4 × 4 in Kaggle Settings, then restart the session.")
if torch.cuda.device_count() != NUM_GPUS:
    raise RuntimeError("GPU selection was applied too late. Restart, then Run All.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This optimizer needs a CUDA GPU with bfloat16 support; choose L4.")
print("Using:", [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)])
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(["nvidia-smi"], check=True)
for package in ("numpy", "numba"):
    if importlib.util.find_spec(package) is None:
        raise RuntimeError(f"Missing {package}; attach an offline wheel. Do not enable Internet for submission.")

flash_probe = [sys.executable, "-c", "from flash_attn import flash_attn_varlen_func; import flash_attn; print(flash_attn.__version__)"]
if ATTENTION_BACKEND == "flash":
    probe = subprocess.run(flash_probe, text=True, capture_output=True)
    if FLASH_ATTN_WHEEL is not None or probe.returncode:
        if FLASH_ATTN_WHEEL is None:
            tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
            wheels = sorted(p for p in Path("/kaggle/input").rglob("*.whl")
                            if p.name.startswith("flash_attn-") and tag in p.name)
            if len(wheels) != 1:
                raise RuntimeError(
                    f"No working FlashAttention import; found {len(wheels)} wheels for {tag}. "
                    "Attach a compatible wheel and set FLASH_ATTN_WHEEL, or set ATTENTION_BACKEND='native'.\n"
                    + probe.stderr[-3000:]
                )
            FLASH_ATTN_WHEEL = wheels[0]
        wheel = Path(FLASH_ATTN_WHEEL)
        if not wheel.is_file() or wheel.suffix != ".whl":
            raise FileNotFoundError(f"Offline wheel not found: {wheel}")
        wheel = restore_wheel_filename(wheel)
        for dependency in sorted(Path(FLASH_ATTN_WHEEL).parent.glob("einops-*.whl")):
            subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                            "--no-deps", str(dependency)], check=True)
        print("Installing offline wheel:", wheel)
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                        "--force-reinstall", str(wheel)], check=True)
        probe = subprocess.run(flash_probe, text=True, capture_output=True)
    if probe.returncode:
        raise RuntimeError("FlashAttention wheel is incompatible with this environment.\n" + probe.stderr[-5000:])
    print("FlashAttention:", probe.stdout.strip())
else:
    print("Using native PyTorch attention; no FlashAttention wheel will be used.")
import torch._dynamo
for option in ("cache_size_limit", "recompile_limit"):
    if hasattr(torch._dynamo.config, option):
        setattr(torch._dynamo.config, option, 64)


## Embedded source

This payload contains the frozen mdlARC runtime and Kaggle submission
helpers. Its checksum is verified before extraction.

This checkpoint has **task_embeddings=False**. Task conditioning stays
disabled during adaptation and decoding. The runtime retains inactive task
tables in its checkpoint format; they are neither used nor trained.
Token embeddings remain frozen. Only architecture-enabled parameters in
transformer blocks **20–23** are trained.


In [ ]:
import base64
import hashlib
import io
import zipfile

RUNTIME_SHA256 = "a154fb83278fec966ac2f9f9300337c853f474711deaeca4da8e6fab6f01364a"
RUNTIME_B64 = "..."
runtime_bytes = base64.b64decode(RUNTIME_B64)
if hashlib.sha256(runtime_bytes).hexdigest() != RUNTIME_SHA256:
    raise RuntimeError("Embedded runtime checksum failed; re-upload the original notebook.")
RUNTIME = WORKING / "mdlarc_runtime"
with zipfile.ZipFile(io.BytesIO(runtime_bytes)) as archive:
    for entry in archive.infolist():
        target = (RUNTIME / entry.filename).resolve()
        target.relative_to(RUNTIME.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(entry))
sys.path.insert(0, str(RUNTIME / "src"))
sys.path.insert(0, str(RUNTIME / "kaggle"))
del runtime_bytes, RUNTIME_B64

if ATTENTION_BACKEND == "native":
    # Ignore an incompatible installed extension as well as attached wheels.
    sys.modules["flash_attn"] = None
    sys.modules["flash_attn_interface"] = None
import build
import tinytransformer
import train
import submission_utils as ku
if ATTENTION_BACKEND == "native":
    tinytransformer.flash_attn_varlen_func = None
    tinytransformer.flash_attn_varlen_qkvpacked_func = None
    tinytransformer.flash_attn_varlen_kvpacked_func = None
elif tinytransformer.flash_attn_varlen_func is None:
    raise RuntimeError("FlashAttention was not imported into the notebook kernel. Restart and Run All.")
print("Runtime SHA-256:", RUNTIME_SHA256)


## Verify inputs and prepare all 240 tasks

Checkpoint size, full SHA-256 and architecture are checked before use.
Both training and decoding use the entire pinned 240-task challenge set.
Only demonstrations supply training targets; queries contain inputs only.
The separate query-answer file is opened after all generation and merging.


In [ ]:
import gc
import json
from time import perf_counter
import multi_gpu as mg

CHECKPOINT_PATH = ku.find_checkpoint(Path("/kaggle/input"), CHECKPOINT_PATH)
TEST_CHALLENGES = ku.find_validation_challenges(Path("/kaggle/input"), TEST_CHALLENGES, VALIDATION_IDENTITY)
print("Verifying checkpoint size and full SHA-256 before load...", flush=True)
checkpoint_hash = ku.verify_checkpoint_file(CHECKPOINT_PATH, CHECKPOINT_IDENTITY)
original = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False, mmap=True)
ku.verify_checkpoint_metadata(original, CHECKPOINT_IDENTITY)
print("Original pretrained epoch:", original["epoch"], "SHA-256:", checkpoint_hash)
print("Architecture:", original["config"])
del original
gc.collect()
data_path = WORKING / "runtime_training_challenges.json"
training_challenges = ku.prepare_challenges(TEST_CHALLENGES, data_path)
decode_data_path = WORKING / "runtime_decode_challenges.json"
challenges = ku.prepare_challenges(TEST_CHALLENGES, decode_data_path,
    smoke_test=SMOKE_TEST, smoke_task_count=NUM_GPUS, smoke_strategy="largest")
print("Training tasks:", len(training_challenges), "demonstration pairs:",
      sum(len(t["train"]) for t in training_challenges.values()))
print("Decode tasks:", list(challenges) if SMOKE_TEST else len(challenges),
      "queries:", sum(len(t["test"]) for t in challenges.values()))
print("Decode batch per GPU:", EVAL_BATCH_SIZE, "prefill token budget:",
      PREFILL_TOKEN_BUDGET, "four-head speculative:", INFERENCE_SPECULATIVE)
print("Frozen source notebook SHA-256:", PINNED_SOURCE_NOTEBOOK_SHA256,
      "packaged runtime SHA-256:", RUNTIME_SHA256)
if (len(training_challenges) != VALIDATION_IDENTITY["tasks"] or
    sum(len(t["train"]) for t in training_challenges.values()) != VALIDATION_IDENTITY["demonstration_pairs"] or
    sum(len(t["test"]) for t in training_challenges.values()) != VALIDATION_IDENTITY["query_pairs"]):
    raise ValueError("Pinned validation dataset count mismatch.")
manifest_path = TEST_CHALLENGES.with_name(VALIDATION_IDENTITY["manifest"]["filename"])
ku.verify_dataset_file(manifest_path, VALIDATION_IDENTITY["manifest"])
validation_groups = json.loads(manifest_path.read_text())["groups"]


## Synchronized DDP adaptation — only blocks 20–23

Five fresh adaptation epochs, with microbatch 2 ×
accumulation 2 × four GPUs = effective batch 16. This test uses 80 training
augmentations and all available demos. A fresh optimizer/schedule is created
from original pretrained weights every time. Architecture-disabled parameters
remain disabled; optimizer groups contain exactly the trainable parameters.

`MDLARC_STARTUP_EVENT first_optimizer_step` confirms a completed update.
Per-rank reports count actual optimizer steps and completed epochs. Liveness
messages only report process health. The eight-hour training ETA guard reserves
time for decoding; a rejected estimate stops the run without changing epochs.
Workers exit completely before inference releases/allocates its CUDA memory.


In [ ]:
SAVED_DEFAULTS = json.loads("{\"seed\": 42, \"optimizer\": \"normuon\", \"FA\": 2, \"normuon_momentum\": 0.95, \"normuon_momentum_min\": 0.85, \"normuon_warmup_steps\": 0, \"normuon_cooldown_steps\": 0, \"normuon_beta2\": 0.95, \"normuon_bucket_size\": 0, \"normuon_qkv_lr_multiplier\": 1.0, \"normuon_attention_lr_multiplier\": 1.0, \"normuon_fc_in_lr_multiplier\": 1.0, \"normuon_fc_out_lr_multiplier\": 1.0, \"normuon_example_value_latent_lr_multiplier\": 1.0, \"normuon_other_lr_multiplier\": 1.0, \"normuon_residual_output_lr_multiplier\": 1.0, \"adamw_decay_lr_multiplier\": 1.0, \"adamw_attention_lr_multiplier\": 1.0, \"adamw_token_embedding_lr_multiplier\": 1.0, \"adamw_task_factor_lr_multiplier\": 1.0, \"adamw_example_value_latent_lr_multiplier\": 1.0, \"adamw_no_decay_lr_multiplier\": 1.0, \"weight_decay\": 0.1, \"attention_weight_decay\": 0.01, \"token_embedding_weight_decay\": 0.01, \"task_embedding_weight_decay\": 0.01, \"grad_clip\": 1.0, \"warmup_pct\": 0.005, \"resume_warmup_pct\": 0.0, \"wsd_decay_start_pct\": 0.8, \"lr_floor\": 0.0001, \"mtp_decay\": [0.8, 0.9, 1.0], \"augmentation_sampling_mode\": \"context_palette\", \"inference_temperature\": null, \"inference_top_k\": null, \"training_demo_schedule\": \"all\", \"inference_demo_policy\": \"leave_one_out\", \"prefill_token_budget\": 4096}")
run_dir = WORKING / "runs" / ("smoke_last4_val240_timing_e5" if SMOKE_TEST else "last4_val240_timing_e5")
run_dir.mkdir(parents=True, exist_ok=True)
# Remove prior stage artifacts so failed reruns cannot accept stale results.
for stale in ["finetuned.pt", "inference.pt", "train-*.json", "decode-*.json", "shard-*.json"]:
    for path in run_dir.glob(stale):
        path.unlink()
save_path = run_dir / "finetuned.pt"
inference_path = run_dir / "inference.pt"
request_path = run_dir / "request.json"
request = {
    "working": str(WORKING), "run_dir": str(run_dir), "num_gpus": NUM_GPUS,
    "checkpoint_path": str(CHECKPOINT_PATH), "checkpoint_identity": CHECKPOINT_IDENTITY,
    "data_path": str(data_path), "decode_data_path": str(decode_data_path),
    "save_path": str(save_path), "inference_checkpoint": str(inference_path),
    "defaults": SAVED_DEFAULTS, "attention_backend": ATTENTION_BACKEND,
    "enable_compile": ENABLE_COMPILE, "train_log_mode": TRAIN_LOG_MODE,
    "progress_interval_seconds": PROGRESS_INTERVAL_SECONDS,
    "smoke_test": SMOKE_TEST, "max_finetune_hours": MAX_FINETUNE_HOURS,
    "trainable_blocks": {"first": TRAIN_FIRST_LAYERS, "last": TRAIN_LAST_LAYERS},
    "eval_augments": 2 if SMOKE_TEST else EVAL_AUGMENTS,
    "eval_batch_size": EVAL_BATCH_SIZE, "prefill_token_budget": PREFILL_TOKEN_BUDGET,
    "inference_speculative": INFERENCE_SPECULATIVE,
    "finetune": {
        "epochs": 1 if SMOKE_TEST else EPOCHS,
        "batch_size": TRAIN_BATCH_SIZE, "accumulation": GRADIENT_ACCUMULATION_STEPS,
        "max_augments": TRAIN_AUGMENTS, "num_workers": NUM_WORKERS,
        "normuon_lr": NORMUON_LR, "adamw_lr": ADAMW_LR,
    },
}
mg.write_json(request_path, request)
print("ADAPTATION REQUEST:", json.dumps(request, indent=2), flush=True)
started = perf_counter()
mg.launch_training(request_path)
adaptation_hours = (perf_counter() - started) / 3600
training_stats = [json.loads((run_dir / f"train-{rank}.json").read_text())
                  for rank in range(NUM_GPUS)]
if not all(s["optimizer_updates"] > 0 and s["epochs"] == request["finetune"]["epochs"]
           and len(s["epoch_seconds"]) == request["finetune"]["epochs"]
           and s["trainable_blocks"] == [20, 21, 22, 23]
           and s["effective_batch_size"] == 16 for s in training_stats):
    raise RuntimeError("Training reports did not verify the complete requested adaptation.")
mg.make_inference_checkpoint(save_path, inference_path, request["finetune"]["epochs"])
print("CHECKPOINT SAVED AND RELOADED:", inference_path)
print("Adaptation hours:", adaptation_hours, "Per-GPU training:", training_stats)


## Independent task-sharded decode, merge and final checks

Four fresh processes, one visible GPU each, reload the adapted model-only
checkpoint. All augmentations of a task stay together for native AAIVR ranking.
Each uses batch 1, budget 4096 and greedy four-head speculative decoding by
default. A single long prompt still needs to fit its GPU; the budget caps
batching and does not truncate prompts. Missing shards, failed workers and
malformed output replacements prevent a successful smoke marker.


In [ ]:
started = perf_counter()
mg.launch_inference(request_path)
output = WORKING / ("smoke_timing240_predictions.json" if SMOKE_TEST else "timing240_predictions.json")
repairs, inference_stats = mg.merge_shards(request, challenges, output)
decoding_hours = (perf_counter() - started) / 3600
summary = {
    "mode": 'validation', "smoke_test": SMOKE_TEST, "checkpoint_sha256": checkpoint_hash,
    "runtime_sha256": RUNTIME_SHA256, "source_notebook_sha256": PINNED_SOURCE_NOTEBOOK_SHA256,
    "torch": torch.__version__, "cuda": torch.version.cuda,
    "attention_backend": ATTENTION_BACKEND, "enable_compile": ENABLE_COMPILE,
    "request": request, "training_tasks": len(training_challenges),
    "training_demonstration_pairs": sum(len(t["train"]) for t in training_challenges.values()),
    "decode_task_ids": list(challenges), "decode_queries": sum(len(t["test"]) for t in challenges.values()),
    "training": training_stats, "inference": inference_stats,
    "adaptation_hours": adaptation_hours, "decoding_hours": decoding_hours,
    "malformed_attempt_replacements": repairs,
}
mg.write_json(WORKING / ("smoke_timing240_run_info.json" if SMOKE_TEST else "timing240_run_info.json"), summary)
print("Parallel decode hours:", decoding_hours, "Per-GPU inference:", inference_stats)
if SMOKE_TEST and repairs:
    raise RuntimeError("Smoke decoding produced malformed attempts; inspect logs before full mode.")
# First answer-file read: parent only, after every decode worker and merge.
solutions_path = TEST_CHALLENGES.with_name(VALIDATION_IDENTITY["solutions"]["filename"])
ku.verify_dataset_file(solutions_path, VALIDATION_IDENTITY["solutions"])
scores = ku.score_validation_groups(json.loads(output.read_text()), challenges,
    json.loads(solutions_path.read_text()), validation_groups)
score_path = WORKING / ("smoke_timing240_scores.json" if SMOKE_TEST else "timing240_scores.json")
mg.write_json(score_path, scores)
for group, result in scores.items():
    print(group, "tasks:", result["tasks"], "queries:", result["queries"],
          "top1/top2 %:", result.get("top1", {}).get("percent"), result.get("top2", {}).get("percent"))
print("VALIDATION COMPLETE:", output, "Scores:", score_path)
if SMOKE_TEST:
    print("SMOKE TEST PASSED: one real epoch, all ranks updated, checkpoint save/reload, four-worker decode/merge.")
    print("Restart the session, set SMOKE_TEST=False. Full mode loads the original epoch330 checkpoint afresh.")
    print("Do not submit smoke outputs.")
query_count = sum(len(t["test"]) for t in challenges.values())
augmented_cases = query_count * request["eval_augments"]
if len(challenges) != 240 or query_count != 336 or augmented_cases != 26880:
    raise RuntimeError("Timing benchmark did not decode the complete requested workload.")
decode_seconds = decoding_hours * 3600
benchmark = {
    "training_epochs": request["finetune"]["epochs"],
    "training_tasks": len(training_challenges), "decode_tasks": len(challenges),
    "decode_queries": query_count, "inference_augments": request["eval_augments"],
    "augmented_query_cases": augmented_cases,
    "adaptation_seconds": adaptation_hours * 3600,
    "decode_wall_seconds": decode_seconds, "decode_wall_hours": decoding_hours,
    "aggregate_augmented_cases_per_second": augmented_cases / decode_seconds,
    "decode_scope": "Worker startup, load, compile, generation, AAIVR and merge; excludes answer scoring.",
    "num_gpus": NUM_GPUS, "decode_batch_per_gpu": EVAL_BATCH_SIZE,
    "prefill_token_budget": PREFILL_TOKEN_BUDGET,
    "inference_speculative": INFERENCE_SPECULATIVE,
    "checkpoint_sha256": checkpoint_hash, "runtime_sha256": RUNTIME_SHA256,
    "training": training_stats, "inference": inference_stats,
}
mg.write_json(WORKING / "timing240_benchmark.json", benchmark)
print("TIMING BENCHMARK COMPLETE")
print(f"Training: {adaptation_hours * 60:.2f} minutes (5 fresh adaptation epochs).")
print(f"Full240 decode: {decode_seconds / 60:.2f} minutes / {decoding_hours:.3f} hours.")
print(f"Aggregate throughput: {augmented_cases / decode_seconds:.3f} augmented query cases/second across four GPUs.")
print("Timing report:", WORKING / "timing240_benchmark.json")
